In [2]:
# =========================
# Task 2: Build per-UMI, gene-aware family table from molecule_info.h5
# =========================
import os, time, math
import numpy as np
import pandas as pd
import polars as pl
import scanpy as sc
import h5py

# --- Paths (reusing your working ones) ---
OUTS_DIR = "/mnt/ribolution/secondary_analysis/2024_SpatSeq_VisiumHD/240726_VH00421_54_2223G3WNX/results-rerun/koeln/C24_13883/outs/"
MOLECULE_INFO = os.path.join(OUTS_DIR, "molecule_info.h5")
MAPPING_PARQUET = os.path.join(OUTS_DIR, "barcode_mappings.parquet")
adata = sc.read_h5ad("/homes/olymp/julius.guntrum/tmp/Koeln/bone/bone_ref.h5ad")
OUT_DIR = "/homes/olymp/julius.guntrum/tmp/Koeln/bone"
# # --- Paths (reusing your working ones) ---


# OUT_DIR = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/stdata/4"
os.makedirs(OUT_DIR, exist_ok=True)

# --- Parameters ---
CHUNK_MOLECULES = 20_000_000            # tune based on RAM; 5M ~ safe
CB_COL = "square_002um"
CELLID_COL = "cell_id"
IN_CELL_COL = "in_cell"
STRIP_CB_SUFFIX = True                  # strip "-1" from parquet barcodes
KEEP_FEATURE_TYPE_SUBSTR = "gene expression"  # case-insensitive substring to keep

SHARD_GLOB = os.path.join(OUT_DIR, "family_geneaware_shard_*.parquet")
MANIFEST = os.path.join(OUT_DIR, "family_geneaware_shards.txt")

# --- Sanity prerequisites ---
assert os.path.exists(MOLECULE_INFO), f"Missing file: {MOLECULE_INFO}"
assert os.path.exists(MAPPING_PARQUET), f"Missing file: {MAPPING_PARQUET}"
assert "adata" in globals(), "Please load your full-depth AnnData into `adata` first."

# ----------------------------
# 1) Build CB (2 µm barcode) → cell_idx map
# ----------------------------
t0 = time.time()
print("[map] loading barcode_mappings.parquet ...")
lf = pl.scan_parquet(MAPPING_PARQUET)
cols = set(lf.collect_schema().names())
for c in [CB_COL, CELLID_COL, IN_CELL_COL]:
    assert c in cols, f"Column '{c}' missing in mapping parquet."

lf2 = lf.filter(pl.col(IN_CELL_COL) == True)
lf2 = lf2.filter(pl.col(CELLID_COL).is_not_null() & (pl.col(CELLID_COL) != ""))

def _strip_dash1(s: str) -> str:
    s = "" if s is None else str(s)
    return s[:-2] if s.endswith("-1") else s

lf2 = lf2.with_columns(
    pl.col(CB_COL).cast(pl.Utf8).map_elements(_strip_dash1, return_dtype=pl.Utf8).alias("CB_norm")
)

mp = lf2.select(["CB_norm", CELLID_COL]).collect()
pairs = mp.to_pandas()

# adata cell index
cellid_to_idx = {str(cid): i for i, cid in enumerate(adata.obs_names.astype(str))}
pairs = pairs[pairs[CELLID_COL].astype(str).isin(cellid_to_idx.keys())]

cb_to_cellidx = {}
for cb, cid in zip(pairs["CB_norm"].tolist(), pairs[CELLID_COL].astype(str).tolist()):
    if cb not in cb_to_cellidx:
        cb_to_cellidx[cb] = cellid_to_idx[cid]

print(f"[map] mapped CB→cell: {len(cb_to_cellidx):,} unique CBs in {time.time()-t0:.1f}s")

# ----------------------------
# 2) Feature (gene) mapping
# ----------------------------
print("[feat] loading features from molecule_info.h5 ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    feat_id = f["features/id"][:]            # bytes, like Ensembl or probe id
    feat_nm = f["features/name"][:]          # bytes, gene symbols
    feat_tp = f["features/feature_type"][:]  # bytes

feat_id = np.array([x.decode("utf-8","ignore") for x in feat_id], dtype=object)
feat_nm = np.array([x.decode("utf-8","ignore") for x in feat_nm], dtype=object)
feat_tp = np.array([x.decode("utf-8","ignore") for x in feat_tp], dtype=object)

# Feature type summary
u, c = np.unique(feat_tp, return_counts=True)
print("[feat] feature_type counts:")
for t, n in zip(u, c):
    print(f"       {t}: {n:,}")

keep_mask_type = np.array(
    [KEEP_FEATURE_TYPE_SUBSTR in t.lower() for t in feat_tp], dtype=bool
)
print(f"[feat] keeping types with '{KEEP_FEATURE_TYPE_SUBSTR}': {keep_mask_type.sum():,}/{feat_tp.size:,}")

# adata gene index maps (symbols + Ensembl versionless)
vn = pd.Index(adata.var_names.astype(str))
var_to_idx = {g: i for i, g in enumerate(vn)}

def _strip_version(x: str) -> str:
    return x.split(".")[0] if "." in x else x

ens_to_idx = {}
for i, g in enumerate(vn):
    if g.startswith("ENSG"):
        ens_to_idx[_strip_version(g)] = i

feat_to_gene_idx = np.full(feat_id.size, -1, dtype=np.int32)
hits_ens = hits_sym = 0
for j in range(feat_id.size):
    if not keep_mask_type[j]:
        continue
    gid = _strip_version(feat_id[j])
    gnm = feat_nm[j]
    gi = -1
    if gid in ens_to_idx:
        gi = ens_to_idx[gid]; hits_ens += 1
    elif gnm in var_to_idx:
        gi = var_to_idx[gnm]; hits_sym += 1
    feat_to_gene_idx[j] = gi

n_unmapped = int((feat_to_gene_idx < 0).sum())
print(f"[feat] mapped features → adata.var_names | Ensembl={hits_ens:,}, Symbol={hits_sym:,}, Unmapped={n_unmapped:,}")

# ----------------------------
# 3) Barcode index → cell_idx vector
# ----------------------------
print("[bar] loading molecule_info barcodes ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    barcodes = f["barcodes"][:]  # bytes
barcodes = [x.decode("utf-8","ignore") for x in barcodes]
n_barcodes = len(barcodes)
barcode_to_cell_idx = np.full(n_barcodes, -1, dtype=np.int32)
hit = 0
for i, s in enumerate(barcodes):
    ci = cb_to_cellidx.get(s, -1)
    barcode_to_cell_idx[i] = ci
    if ci >= 0: hit += 1
print(f"[bar] barcodes mapped to any cell: {hit:,} / {n_barcodes:,}")

# ----------------------------
# 4) Stream molecule_info rows → per-UMI family shards
# ----------------------------
print("[fam] streaming molecules to per-UMI shards ...")
shards = []
with h5py.File(MOLECULE_INFO, "r") as f:
    n_mol = f["barcode_idx"].shape[0]
    n_chunks = math.ceil(n_mol / CHUNK_MOLECULES)
    print(f"[fam] molecules: {n_mol:,} | chunks: {n_chunks}")

    for k in range(n_chunks):
        start = k * CHUNK_MOLECULES
        end   = min((k+1) * CHUNK_MOLECULES, n_mol)
        t1 = time.time()

        bc_idx  = f["barcode_idx"][start:end].astype(np.int64)
        ft_idx  = f["feature_idx"][start:end].astype(np.int64)
        umi     = f["umi"][start:end].astype(np.uint32)
        reads   = f["count"][start:end].astype(np.int32)

        # map indices
        cells = barcode_to_cell_idx[bc_idx]
        genes = feat_to_gene_idx[ft_idx]

        # keep if: on-cell, mapped gene, positive reads
        mask = (cells >= 0) & (genes >= 0) & (reads > 0)
        kept = int(mask.sum())

        if kept > 0:
            df_pl = pl.DataFrame({
                "cell_idx":   pl.Series(cells[mask].astype(np.int32)),
                "gene_idx":   pl.Series(genes[mask].astype(np.int32)),
                "umi_reads":  pl.Series(reads[mask].astype(np.int32)),
                "umi_id":     pl.Series(umi[mask].astype(np.uint32)),  # optional, useful for QA
            })
            shard = os.path.join(OUT_DIR, f"family_geneaware_shard_{k:04d}.parquet")
            df_pl.write_parquet(shard, compression="zstd")
            shards.append(shard)

        rate = (end - start) / max(1e-6, (time.time() - t1))
        print(f"[fam] chunk {k+1}/{n_chunks} rows {start:,}..{end-1:,} | kept={kept:,} | rate≈{rate:,.0f}/s")

if not shards:
    raise RuntimeError("No per-UMI families were collected; check mapping/filters.")

# Write a manifest so downstream can glob safely (and you have a record)
with open(MANIFEST, "w") as fh:
    for s in shards:
        fh.write(s + "\n")

# Quick QC via lazy scan (no full load)
lf_all = pl.scan_parquet(SHARD_GLOB)
qc = lf_all.select([
    pl.len().alias("n_families"),
    pl.col("umi_reads").sum().alias("total_reads"),
    pl.col("cell_idx").n_unique().alias("n_cells_nonempty"),
    pl.col("gene_idx").n_unique().alias("n_genes_nonempty"),
    pl.col("umi_reads").mean().alias("mean_reads_per_family"),
]).collect().to_pandas().iloc[0].to_dict()

print("[fam][QC]", {k: (int(v) if isinstance(v, (np.integer, int)) else (round(float(v),3))) for k,v in qc.items()})
print("[done] shards written. Manifest:", MANIFEST)


[map] loading barcode_mappings.parquet ...
[map] mapped CB→cell: 419,337 unique CBs in 4.9s
[feat] loading features from molecule_info.h5 ...
[feat] feature_type counts:
       Gene Expression: 39,126
[feat] keeping types with 'gene expression': 39,126/39,126
[feat] mapped features → adata.var_names | Ensembl=12,979, Symbol=25,666, Unmapped=481
[bar] loading molecule_info barcodes ...
[bar] barcodes mapped to any cell: 419,337 / 11,222,500
[fam] streaming molecules to per-UMI shards ...
[fam] molecules: 2,111,502 | chunks: 1
[fam] chunk 1/1 rows 0..2,111,501 | kept=1,368,465 | rate≈3,286,296/s
[fam][QC] {'n_families': 1368465.0, 'total_reads': 10670749.0, 'n_cells_nonempty': 9125.0, 'n_genes_nonempty': 16846.0, 'mean_reads_per_family': 7.798}
[done] shards written. Manifest: /homes/olymp/julius.guntrum/tmp/Koeln/bone/family_geneaware_shards.txt


In [5]:
# ============================================
# STEP 2b — Corrected subsampling with REPLICATES
#           RPC clamped to feasible max; N_STEPS across range
#           Save N_REPS per target to account for sampling variability
# ============================================
import os, time, json, hashlib
import numpy as np
import pandas as pd
import polars as pl
import scanpy as sc
from scipy import sparse

# -------- CONFIG (edit if needed) --------
OUT_ROOT   = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2"
MANIFEST   = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/stdata/4/family_geneaware_shards.txt"
REF_H5AD   = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/stdata/2/merged.h5ad"

SAVE_DIR   = os.path.join(OUT_ROOT, "runs_rpc_clamped")   # output folder
os.makedirs(SAVE_DIR, exist_ok=True)

LAYER_LOGNORM = "lognorm"

# grid settings
N_STEPS = 10        # equally spaced RPC targets (incl. max)
N_REPS  = 3         # <-- number of replicate subsamples PER target RPC

# random seeds: base + per(target,rep) offset
RNG_SEED_BASE = 12345

# ----------------------------------------

def _ensure_lognorm(ad, layer=LAYER_LOGNORM):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=1e4, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _read_shard(path):
    """Expect columns: cell_idx, gene_idx, umi_reads  (or legacy family_reads)."""
    try:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","umi_reads"]).to_pandas()
    except Exception:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","family_reads"]).to_pandas()
        df.rename(columns={"family_reads":"umi_reads"}, inplace=True)
    # dtypes
    df["cell_idx"]  = df["cell_idx"].astype(np.int32, copy=False)
    df["gene_idx"]  = df["gene_idx"].astype(np.int32, copy=False)
    df["umi_reads"] = df["umi_reads"].astype(np.int32, copy=False)
    return df

def _build_counts_matrix(n_cells, n_genes, grouped_triplets):
    if not grouped_triplets:
        return sparse.csr_matrix((n_cells, n_genes), dtype=np.int32)
    rows = np.concatenate([t[0] for t in grouped_triplets])
    cols = np.concatenate([t[1] for t in grouped_triplets])
    dat  = np.concatenate([t[2] for t in grouped_triplets])
    return sparse.coo_matrix((dat, (rows, cols)), shape=(n_cells, n_genes), dtype=np.int32).tocsr()

def _load_manifest(p):
    with open(p, "r") as f:
        paths = [ln.strip() for ln in f if ln.strip()]
    if not paths:
        raise RuntimeError(f"Empty manifest: {p}")
    return paths

def _seed_for(target_rpc:int, rep:int) -> np.random.Generator:
    # stable per (target_rpc, rep) RNG using base + hash; reproducible across runs
    h = int(hashlib.blake2b(f"{target_rpc}|{rep}".encode(), digest_size=8).hexdigest(), 16) & 0x7FFFFFFF
    return np.random.default_rng(RNG_SEED_BASE + h)

# ---------- Load reference + shards ----------
ad_ref = sc.read_h5ad(REF_H5AD)
n_cells, n_genes = ad_ref.n_obs, ad_ref.n_vars
print(f"[ref] {n_cells:,} cells × {n_genes:,} genes")

shards = _load_manifest(MANIFEST)
print(f"[manifest] {len(shards)} shards")

# ---------- Compute total reads present in shards (denominator we actually thin) ----------
reads_in_shards = 0
families_in_shards = 0
t0 = time.time()
for i, sh in enumerate(shards, 1):
    try:
        s = pl.read_parquet(sh, columns=["umi_reads"]).select(pl.col("umi_reads").sum()).item()
    except Exception:
        s = pl.read_parquet(sh, columns=["family_reads"]).select(pl.col("family_reads").sum()).item()
    reads_in_shards += int(s)

    fam_ct = pl.read_parquet(sh, columns=["cell_idx"]).height
    families_in_shards += int(fam_ct)
    if i == 1 or i % 2 == 0 or i == len(shards):
        print(f"  [scan] shard {i:02d}/{len(shards)} | +reads={int(s):,} | cum_reads={reads_in_shards:,}")
print(f"[totals] reads_in_shards={reads_in_shards:,} | families={families_in_shards:,} | {time.time()-t0:.1f}s")

# ---------- Determine feasible max RPC and build equally spaced target list ----------
max_rpc = int(np.floor(reads_in_shards / n_cells))
print(f"[feasible] max_rpc={max_rpc:,} (reads_in_shards {reads_in_shards:,} / cells {n_cells:,})")

if max_rpc <= 0:
    target_rpcs = []
else:
    # Equally spaced integers from 50 to max_rpc (inclusive), with N_STEPS points.
    target_rpcs = sorted(np.unique(np.linspace(50, max_rpc, N_STEPS, dtype=int)).tolist())
    target_rpcs = [t for t in target_rpcs if t > 0]

print("[targets]", target_rpcs)

# ---------- Subsampling loop with REPLICATES ----------
index_rows = []

for target_rpc in target_rpcs:
    # fixed keep-rate for this target (same across reps)
    p_keep = float(target_rpc) * float(n_cells) / float(reads_in_shards)
    p_keep = float(np.clip(p_keep, 0.0, 1.0))
    tag    = f"rpc{target_rpc:d}"

    for rep in range(N_REPS):
        out_path = os.path.join(SAVE_DIR, f"subsample_MAPPED_{tag}_r{rep}.h5ad")
        if os.path.exists(out_path):
            print(f"[skip] exists → {out_path}")
            index_rows.append(dict(target_rpc=target_rpc, rep=rep, h5ad=out_path, p_keep=p_keep))
            continue

        rng = _seed_for(target_rpc, rep)
        print(f"\n[run] target_rpc={target_rpc} | rep={rep} | p_keep≈{p_keep:.6g}")
        t1 = time.time()

        grouped = []
        reads_kept_per_cell = np.zeros(n_cells, dtype=np.int64)
        umis_surv_per_cell  = np.zeros(n_cells, dtype=np.int64)

        kept_reads_total = 0
        kept_umis_total  = 0

        for si, sh in enumerate(shards, 1):
            s0 = time.time()
            df  = _read_shard(sh)
            r   = df["umi_reads"].to_numpy()
            cel = df["cell_idx"].to_numpy()
            gen = df["gene_idx"].to_numpy()

            kept = rng.binomial(r, p_keep).astype(np.int32)
            survive = kept > 0

            # per-cell kept reads
            if kept.size:
                pc_reads = (
                    pd.DataFrame({"cell_idx": cel, "kept": kept})
                    .groupby("cell_idx", sort=False)["kept"].sum().reset_index()
                )
                reads_kept_per_cell[pc_reads["cell_idx"].to_numpy()] += pc_reads["kept"].to_numpy(np.int64)
                kept_reads_total += int(pc_reads["kept"].sum())

            # surviving families contribute 1 UMI to their gene
            n_eff = 0
            if survive.any():
                sel_cells = cel[survive]
                sel_genes = gen[survive]
                n_eff = sel_cells.size

                df_chunk = pd.DataFrame({"cell_idx": sel_cells, "gene_idx": sel_genes, "umis": 1}, dtype=np.int32)
                agg = df_chunk.groupby(["cell_idx","gene_idx"], sort=False)["umis"].sum().reset_index()
                grouped.append((agg["cell_idx"].to_numpy(np.int32),
                                agg["gene_idx"].to_numpy(np.int32),
                                agg["umis"].to_numpy(np.int32)))

                pc_umis = (
                    pd.DataFrame({"cell_idx": sel_cells, "one": 1})
                    .groupby("cell_idx", sort=False)["one"].sum().reset_index()
                )
                umis_surv_per_cell[pc_umis["cell_idx"].to_numpy()] += pc_umis["one"].to_numpy(np.int64)
                kept_umis_total += int(pc_umis["one"].sum())

            dt = time.time() - s0
            print(f"  [shard {si:02d}/{len(shards)}] kept_reads={int(pc_reads['kept'].sum()) if kept.size else 0:,} | kept_UMIs={n_eff:,} | {dt:.1f}s")

        # Build counts + AnnData
        X = _build_counts_matrix(n_cells, n_genes, grouped)
        ad = sc.AnnData(
            X=X,
            obs=ad_ref.obs.copy(),
            var=ad_ref.var.copy(),
            obsm=ad_ref.obsm.copy() if ad_ref.obsm is not None else None,
            uns=ad_ref.uns.copy() if ad_ref.uns is not None else None,
        )

        # Attach stats (column names unchanged for compatibility)
        ad.obs[f"reads_sim_MAPPED_{tag}"] = reads_kept_per_cell
        ad.obs[f"umis_sim_MAPPED_{tag}"]  = umis_surv_per_cell
        with np.errstate(divide="ignore", invalid="ignore"):
            sat = 1.0 - (umis_surv_per_cell.astype(float) / np.clip(reads_kept_per_cell, 1, None))
            sat[reads_kept_per_cell == 0] = np.nan
        ad.obs[f"saturation_sim_MAPPED_{tag}"] = sat

        # provenance for later analysis
        ad.uns.setdefault("subsample_meta", {})
        ad.uns["subsample_meta"].update({
            "target_rpc": int(target_rpc),
            "rep": int(rep),
            "p_keep": float(p_keep),
            "reads_in_shards": int(reads_in_shards),
            "n_cells": int(n_cells),
            "n_genes": int(n_genes),
            "seed_base": int(RNG_SEED_BASE),
        })

        _ensure_lognorm(ad, LAYER_LOGNORM)
        ad.write_h5ad(out_path)
        print(f"[done] wrote {out_path} | kept_reads={kept_reads_total:,} | kept_UMIs={kept_umis_total:,} | {time.time()-t1:.1f}s")

        index_rows.append(dict(target_rpc=target_rpc, rep=rep, h5ad=out_path, p_keep=p_keep))

# Save a small index for discovery later (now includes 'rep')
index_csv = os.path.join(SAVE_DIR, "subsample_runs_index.csv")
pd.DataFrame(index_rows).sort_values(["target_rpc","rep"]).to_csv(index_csv, index=False)
print(f"\n[save] run index → {index_csv}")
print("[note] Largest target_rpc equals floor(total_reads_in_shards / n_cells); larger targets are skipped.")


[ref] 58,243 cells × 38,645 genes
[manifest] 10 shards
  [scan] shard 01/10 | +reads=16,842,006 | cum_reads=16,842,006
  [scan] shard 02/10 | +reads=17,381,239 | cum_reads=34,223,245
  [scan] shard 04/10 | +reads=16,756,693 | cum_reads=68,630,091
  [scan] shard 06/10 | +reads=17,152,908 | cum_reads=101,487,320
  [scan] shard 08/10 | +reads=17,274,449 | cum_reads=135,628,080
  [scan] shard 10/10 | +reads=2,699,178 | cum_reads=156,709,756
[totals] reads_in_shards=156,709,756 | families=36,359,000 | 1.1s
[feasible] max_rpc=2,690 (reads_in_shards 156,709,756 / cells 58,243)
[targets] [50, 343, 636, 930, 1223, 1516, 1810, 2103, 2396, 2690]

[run] target_rpc=50 | rep=0 | p_keep≈0.0185831
  [shard 01/10] kept_reads=312,418 | kept_UMIs=297,786 | 0.5s
  [shard 02/10] kept_reads=323,381 | kept_UMIs=307,995 | 0.4s
  [shard 03/10] kept_reads=328,531 | kept_UMIs=312,892 | 0.4s
  [shard 04/10] kept_reads=311,533 | kept_UMIs=296,646 | 0.3s
  [shard 05/10] kept_reads=291,777 | kept_UMIs=277,951 | 0.3s

/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc50_r0.h5ad | kept_reads=2,912,250 | kept_UMIs=2,774,506 | 4.7s

[run] target_rpc=50 | rep=1 | p_keep≈0.0185831
  [shard 01/10] kept_reads=312,037 | kept_UMIs=297,454 | 0.3s
  [shard 02/10] kept_reads=323,949 | kept_UMIs=308,598 | 0.4s
  [shard 03/10] kept_reads=328,185 | kept_UMIs=312,391 | 0.3s
  [shard 04/10] kept_reads=312,116 | kept_UMIs=297,272 | 0.3s
  [shard 05/10] kept_reads=291,568 | kept_UMIs=277,704 | 0.3s
  [shard 06/10] kept_reads=319,340 | kept_UMIs=304,186 | 0.3s
  [shard 07/10] kept_reads=313,816 | kept_UMIs=298,921 | 0.3s
  [shard 08/10] kept_reads=320,722 | kept_UMIs=305,643 | 0.3s
  [shard 09/10] kept_reads=341,863 | kept_UMIs=325,680 | 0.4s
  [shard 10/10] kept_reads=50,131 | kept_UMIs=47,781 | 0.1s


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc50_r1.h5ad | kept_reads=2,913,727 | kept_UMIs=2,775,630 | 4.0s

[run] target_rpc=50 | rep=2 | p_keep≈0.0185831
  [shard 01/10] kept_reads=312,526 | kept_UMIs=298,024 | 0.4s
  [shard 02/10] kept_reads=321,948 | kept_UMIs=306,744 | 0.3s
  [shard 03/10] kept_reads=329,044 | kept_UMIs=313,197 | 0.4s
  [shard 04/10] kept_reads=310,964 | kept_UMIs=295,855 | 0.3s
  [shard 05/10] kept_reads=291,728 | kept_UMIs=278,040 | 0.3s
  [shard 06/10] kept_reads=318,579 | kept_UMIs=303,683 | 0.3s
  [shard 07/10] kept_reads=313,554 | kept_UMIs=298,735 | 0.3s
  [shard 08/10] kept_reads=320,715 | kept_UMIs=305,762 | 0.3s
  [shard 09/10] kept_reads=341,497 | kept_UMIs=325,330 | 0.4s
  [shard 10/10] kept_reads=50,036 | kept_UMIs=47,681 | 0.1s


...

[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc2690_r0.h5ad | kept_reads=156,673,880 | kept_UMIs=36,357,705 | 17.5s

[run] target_rpc=2690 | rep=1 | p_keep≈0.99977
  [shard 01/10] kept_reads=16,838,080 | kept_UMIs=3,939,006 | 1.3s
  [shard 02/10] kept_reads=17,377,132 | kept_UMIs=4,047,437 | 1.5s
  [shard 03/10] kept_reads=17,646,113 | kept_UMIs=4,075,597 | 1.5s
  [shard 04/10] kept_reads=16,752,770 | kept_UMIs=3,860,210 | 1.3s
  [shard 05/10] kept_reads=15,700,807 | kept_UMIs=3,636,942 | 1.2s
  [shard 06/10] kept_reads=17,148,881 | kept_UMIs=4,004,929 | 1.4s
  [shard 07/10] kept_reads=16,862,487 | kept_UMIs=3,910,368 | 1.4s
  [shard 08/10] kept_reads=17,270,339 | kept_UMIs=4,011,757 | 1.5s
  [shard 09/10] kept_reads=18,378,326 | kept_UMIs=4,246,736 | 1.9s
  [shard 10/10] kept_reads=2,698,574 | kept_UMIs=624,682 | 0.2s


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc2690_r1.h5ad | kept_reads=156,673,509 | kept_UMIs=36,357,664 | 17.6s

[run] target_rpc=2690 | rep=2 | p_keep≈0.99977
  [shard 01/10] kept_reads=16,838,087 | kept_UMIs=3,939,021 | 1.4s
  [shard 02/10] kept_reads=17,377,171 | kept_UMIs=4,047,458 | 1.4s
  [shard 03/10] kept_reads=17,645,990 | kept_UMIs=4,075,587 | 1.4s
  [shard 04/10] kept_reads=16,752,892 | kept_UMIs=3,860,232 | 1.3s
  [shard 05/10] kept_reads=15,700,691 | kept_UMIs=3,636,930 | 1.2s
  [shard 06/10] kept_reads=17,148,929 | kept_UMIs=4,004,910 | 1.4s
  [shard 07/10] kept_reads=16,862,438 | kept_UMIs=3,910,368 | 1.3s
  [shard 08/10] kept_reads=17,270,468 | kept_UMIs=4,011,754 | 1.4s
  [shard 09/10] kept_reads=18,378,376 | kept_UMIs=4,246,752 | 1.9s
  [shard 10/10] kept_reads=2,698,544 | kept_UMIs=624,685 | 0.2s


/homes/olymp/julius.guntrum/.conda/envs/visium_readsub/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


[done] wrote /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc2690_r2.h5ad | kept_reads=156,673,586 | kept_UMIs=36,357,697 | 17.4s

[save] run index → /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_runs_index.csv
[note] Largest target_rpc equals floor(total_reads_in_shards / n_cells); larger targets are skipped.


In [8]:
# ============================================
# STEP 3 — Metrics + plots for RPC-clamped runs (replicate-aware)
# - Works with multiple repetitions per target_rpc (column 'rep' in index CSV)
# - Saves per-rep metrics and aggregated (mean±SD across reps)
# - Plots show per-rep scatter + mean±SD error bars
# ============================================
import os, json, math
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

# -------- CONFIG --------
OUT_ROOT    = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2"
RUNS_DIR    = os.path.join(OUT_ROOT, "runs_rpc_clamped")      # from previous step
METRICS_DIR = os.path.join(OUT_ROOT, "metrics_rpc_clamped")   # new
os.makedirs(METRICS_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"

runs = pd.read_csv(index_csv)
# Normalize dtypes / required columns
runs["target_rpc"] = runs["target_rpc"].astype(int)
if "rep" not in runs.columns:
    runs["rep"] = 0
runs["rep"] = runs["rep"].astype(int)

print("[index]")
print(runs.sort_values(["target_rpc","rep"]))

def _per_cell_genes(ad):
    X = ad.X
    if sparse.issparse(X):
        return np.asarray(X.getnnz(axis=1)).ravel()
    return (np.asarray(X) > 0).sum(axis=1)

def _metrics_for_run(h5ad_path, target_rpc, rep):
    ad = sc.read_h5ad(h5ad_path)
    tag = f"rpc{int(target_rpc)}"
    rcol = f"reads_sim_MAPPED_{tag}"
    ucol = f"umis_sim_MAPPED_{tag}"
    scol = f"saturation_sim_MAPPED_{tag}"

    # Basic guards
    for c in (rcol, ucol, scol):
        if c not in ad.obs.columns:
            raise RuntimeError(f"Missing expected column in {os.path.basename(h5ad_path)}: {c}")

    reads_per_cell = np.asarray(ad.obs[rcol]).astype(float)
    umis_per_cell  = np.asarray(ad.obs[ucol]).astype(float)
    genes_per_cell = _per_cell_genes(ad).astype(float)
    sat_per_cell   = np.asarray(ad.obs[scol]).astype(float)

    out = dict(
        target_rpc=int(target_rpc),
        rep=int(rep),
        n_cells=int(ad.n_obs),
        total_reads=float(np.nansum(reads_per_cell)),
        total_umis=float(np.nansum(umis_per_cell)),
        median_reads=float(np.nanmedian(reads_per_cell)),
        mean_reads=float(np.nanmean(reads_per_cell)),
        median_umis=float(np.nanmedian(umis_per_cell)),
        mean_umis=float(np.nanmean(umis_per_cell)),
        median_genes=float(np.nanmedian(genes_per_cell)),
        mean_genes=float(np.nanmean(genes_per_cell)),
        median_saturation=float(np.nanmedian(sat_per_cell)),
        mean_saturation=float(np.nanmean(sat_per_cell)),
        h5ad=h5ad_path,
    )
    # lightweight consistency check (not fatal)
    try:
        umis_from_X = float(np.asarray(ad.X.sum(axis=1)).ravel().sum())
        if abs(umis_from_X - out["total_umis"]) > 0.01 * max(1.0, out["total_umis"]):
            print(f"[warn] UMIs mismatch (X vs obs) for {os.path.basename(h5ad_path)}: "
                  f"{umis_from_X:.0f} vs {out['total_umis']:.0f}")
    except Exception as e:
        print(f"[note] skipped UMIs-from-X check: {e}")

    return out

# -------- compute per-rep metrics over all runs --------
rows = []
for _, r in runs.sort_values(["target_rpc","rep"]).iterrows():
    print(f"[metrics] rpc={int(r['target_rpc'])} rep={int(r['rep'])}  {os.path.basename(r['h5ad'])}")
    rows.append(_metrics_for_run(r["h5ad"], r["target_rpc"], r["rep"]))

perrep_df = pd.DataFrame(rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)
perrep_df["reads_millions"] = perrep_df["total_reads"] / 1e6

# -------- aggregate across replicates (mean ± SD) --------
def _agg_mean_sd(df, cols, xcol="reads_millions"):
    g = df.groupby("target_rpc", as_index=False)
    base = g[xcol].agg(x_mean="mean", x_sd="std")
    out = base.copy()
    for c in cols:
        agg = g[c].agg(**{f"{c}_mean": "mean", f"{c}_sd": "std"})
        out = out.merge(agg, on="target_rpc", how="left")
    # also keep n
    out = out.merge(g.size().rename(columns={"size":"n"}), on="target_rpc", how="left")
    return out.sort_values("x_mean")

metric_cols = [
    "median_reads","median_umis","median_genes","median_saturation",
    "mean_reads","mean_umis","mean_genes","mean_saturation",
    "total_reads","n_cells"
]
agg_df = _agg_mean_sd(perrep_df, metric_cols, xcol="reads_millions")
agg_df["reads_millions_mean"] = agg_df["x_mean"]
agg_df["reads_millions_sd"]   = agg_df["x_sd"]
agg_df = agg_df.drop(columns=["x_mean","x_sd"])

# -------- save CSVs (per-rep + aggregated) --------
perrep_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped_perrep.csv")
agg_csv    = os.path.join(METRICS_DIR, "metrics_rpc_clamped_agg.csv")
# For backward compatibility, also write the original filename pointing to the aggregated table
compat_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped.csv")

perrep_df.to_csv(perrep_csv, index=False)
agg_df.to_csv(agg_csv, index=False)
agg_df.to_csv(compat_csv, index=False)

print(f"[save] per-rep metrics   → {perrep_csv}")
print(f"[save] aggregated metrics→ {agg_csv}")
print(f"[save] (compat alias)   → {compat_csv}")

# -------- plotting (medians; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots = [
    ("median_reads_mean",      "median_reads_sd",      "Median reads/cell"),
    ("median_umis_mean",       "median_umis_sd",       "Median UMIs/cell"),
    ("median_genes_mean",      "median_genes_sd",      "Median genes/cell"),
    ("median_saturation_mean", "median_saturation_sd", "Median saturation"),
]
for i,(col_mean,col_sd,label) in enumerate(plots, 1):
    ax = plt.subplot(2,2,i)
    # scatter per-rep
    plt.scatter(perrep_df["reads_millions"], perrep_df[col_mean.replace("_mean","")], s=15, alpha=0.4)
    # error bars (mean±SD)
    ax.errorbar(agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
                fmt='-o', capsize=3)
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): medians vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_med_png = os.path.join(METRICS_DIR, "panel_medians_vs_total_reads.png")
plt.savefig(panel_med_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_med_png}")

# -------- plotting (means; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots_mean = [
    ("mean_reads_mean",      "mean_reads_sd",      "Mean reads/cell"),
    ("mean_umis_mean",       "mean_umis_sd",       "Mean UMIs/cell"),
    ("mean_genes_mean",      "mean_genes_sd",      "Mean genes/cell"),
    ("mean_saturation_mean", "mean_saturation_sd", "Mean saturation"),
]
for i,(col_mean,col_sd,label) in enumerate(plots_mean, 1):
    ax = plt.subplot(2,2,i)
    plt.scatter(perrep_df["reads_millions"], perrep_df[col_mean.replace("_mean","")], s=15, alpha=0.4)
    ax.errorbar(agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
                fmt='-o', capsize=3)
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): means vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_mean_png = os.path.join(METRICS_DIR, "panel_means_vs_total_reads.png")
plt.savefig(panel_mean_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_mean_png}")

# -------- artifact schema to help future loading --------
schema = dict(
    runs_index=dict(
        path=index_csv,
        columns=list(pd.read_csv(index_csv, nrows=1).columns),
        description="Index of subsample runs (target_rpc, p_keep, h5ad path, rep).",
        reads_col_pattern="reads_sim_MAPPED_rpc{target_rpc}",
        umis_col_pattern="umis_sim_MAPPED_rpc{target_rpc}",
        saturation_col_pattern="saturation_sim_MAPPED_rpc{target_rpc}",
        has_replicates=True,
        rep_column="rep",
    ),
    metrics=dict(
        per_rep_path=perrep_csv,
        aggregated_path=agg_csv,
        columns_per_rep=list(perrep_df.columns),
        columns_aggregated=list(agg_df.columns),
        description="Per-replicate metrics and aggregated mean±SD over replicates.",
        x_axis_aggregated="reads_millions_mean",
        primary_metrics=[
            # medians
            "median_reads_mean","median_umis_mean","median_genes_mean","median_saturation_mean",
            # means
            "mean_reads_mean","mean_umis_mean","mean_genes_mean","mean_saturation_mean",
        ],
    ),
    figures=[
        panel_med_png,
        panel_mean_png,
    ],
)
schema_json = os.path.join(METRICS_DIR, "artifact_schema.json")
with open(schema_json, "w") as f:
    json.dump(schema, f, indent=2)
print(f"[save] schema → {schema_json}")

print("\n[done] Replicate-aware metrics computed and plots saved.")


[index]
    target_rpc  rep                                               h5ad  \
0           50    0  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
1           50    1  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
2           50    2  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
3          343    0  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
4          343    1  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
5          343    2  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
6          636    0  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
7          636    1  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
8          636    2  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
9          930    0  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
10         930    1  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
11         930    2  /homes/olymp/julius.guntrum/tmp/Koeln/duo/3/te...   
12        1223    0  /homes/ol

In [4]:
# === Cell 1 (replicate-compatible): setup, robust schema loader, prepare reference, define stable params ===
import os, json, gc, warnings, re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

# -------------------- CONFIG (edit if needed) --------------------
# Use the same paths as before
SCHEMA_JSON   = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/metrics_rpc_clamped/artifact_schema.json"
ANALYSIS_ROOT = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/analysis_rpc_clamped"
FIG_DIR       = os.path.join(ANALYSIS_ROOT, "figures")

# Compute settings (stable across runs/samples)
N_PCS         = 30
N_NEI         = 20            # for neighbors & kNN Jaccard graphs later
KNN_OVERLAP_K = 15            # classic value used previously for neighbor-overlap
LEIDEN_RES    = 0.6
RANDOM_STATE  = 0
N_JOBS        = max(1, int(os.environ.get("OMP_NUM_THREADS", "8")))
DO_CAR_ANALYSIS = True        # <-- make CAR+ analysis optional

# Try GPU UMAP if available
_USE_CUML = False
try:
    from cuml.manifold import UMAP as cuUMAP  # noqa: F401
    _USE_CUML = True
except Exception:
    _USE_CUML = False

sc.settings.verbosity = 2
warnings.filterwarnings("ignore", category=UserWarning)
os.makedirs(ANALYSIS_ROOT, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# -------------------- SCHEMA / IO --------------------
def _coerce_schema(raw: dict):
    """
    Accepts either:
      A) Your artifact schema (with top-level 'runs_index'):
         {
           "runs_index": {"path": "...", "columns": [...],
                          "reads_col_pattern": "...", "umis_col_pattern": "...",
                          "saturation_col_pattern": "..."},
           "metrics": {...}  # optional
         }
      B) Analysis schema variant (with 'inputs' / 'outputs'):
         { "inputs": { "runs_index_csv": "..." }, "outputs": {...} }
    """
    out = {}
    if isinstance(raw, dict) and "runs_index" in raw:
        ri = raw["runs_index"]
        out["runs_index_path"]       = ri["path"]
        out["runs_index_columns"]    = ri.get("columns", [])
        out["reads_col_pattern"]     = ri.get("reads_col_pattern")
        out["umis_col_pattern"]      = ri.get("umis_col_pattern")
        out["saturation_col_pattern"]= ri.get("saturation_col_pattern")
        if "metrics" in raw:
            out["metrics_path"]    = raw["metrics"].get("path")
            out["metrics_columns"] = raw["metrics"].get("columns", [])
        return out

    if isinstance(raw, dict) and "inputs" in raw and isinstance(raw["inputs"], dict):
        inp = raw["inputs"]
        if "runs_index_csv" in inp:
            out["runs_index_path"] = inp["runs_index_csv"]
        elif "runs_index" in inp:
            out["runs_index_path"] = inp["runs_index"]
        else:
            raise KeyError("Schema 'inputs' present but missing 'runs_index_csv' or 'runs_index'.")
        out["reads_col_pattern"]     = inp.get("reads_col_pattern")
        out["umis_col_pattern"]      = inp.get("umis_col_pattern")
        out["saturation_col_pattern"]= inp.get("saturation_col_pattern")
        if "outputs" in raw and isinstance(raw["outputs"], dict):
            outs = raw["outputs"]
            out["metrics_path"] = outs.get("clustering_metrics_csv")
        return out

    raise KeyError(f"Could not find runs_index path. Top-level keys: {list(raw.keys())}")

def load_schema(schema_path: str):
    with open(schema_path, "r") as f:
        raw = json.load(f)
    sch = _coerce_schema(raw)
    if not os.path.exists(sch["runs_index_path"]):
        raise FileNotFoundError(f"runs_index not found: {sch['runs_index_path']}")
    return sch

def read_runs_index(schema: dict) -> pd.DataFrame:
    """
    Expect columns in CSV (replicates supported):
      required:  target_rpc, h5ad
      optional:  p_keep, rep
    If 'rep' is missing, it will be added and set to 0.
    """
    path = schema["runs_index_path"]
    df = pd.read_csv(path)

    if "target_rpc" not in df.columns:
        cand = [c for c in df.columns if "target" in c.lower() and "rpc" in c.lower()]
        assert cand, f"No 'target_rpc' column in {path}"
        df = df.rename(columns={cand[0]: "target_rpc"})
    if "h5ad" not in df.columns:
        cand = [c for c in df.columns if "h5ad" in c.lower()]
        assert cand, f"No 'h5ad' column in {path}"
        df = df.rename(columns={cand[0]: "h5ad"})
    if "p_keep" not in df.columns:
        df["p_keep"] = np.nan
    if "rep" not in df.columns:
        df["rep"] = 0

    df["target_rpc"] = pd.to_numeric(df["target_rpc"], errors="coerce").astype("Int64")
    df["rep"]        = pd.to_numeric(df["rep"], errors="coerce").fillna(0).astype(int)
    df = df.dropna(subset=["target_rpc", "h5ad"]).reset_index(drop=True)
    df["h5ad"] = df["h5ad"].astype(str)
    return df.sort_values(["target_rpc","rep"]).reset_index(drop=True)

# -------------------- small helpers --------------------
def _ensure_lognorm(ad, layer="lognorm", target_sum=1e4):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=target_sum, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _compute_hvgs_on_ref(ad_ref, n_top=2000, layer="lognorm"):
    sc.pp.highly_variable_genes(ad_ref, n_top_genes=n_top, flavor="seurat", layer=layer, subset=False)
    if "highly_variable" not in ad_ref.var or ad_ref.var["highly_variable"].sum() == 0:
        raise RuntimeError("Failed to compute HVGs on reference.")
    return ad_ref.var["highly_variable"].values

def _select_gene_id_column(var_df: pd.DataFrame) -> str | None:
    """
    Choose a stable feature ID column (prefer Ensembl-like) for cross-run alignment.
    Priority list; first column with unique, non-null values across all vars wins.
    """
    candidates = [
        "gene_ids", "gene_id", "feature_id", "ensembl", "ensembl_id", "ensg", "id"
    ] + [c for c in var_df.columns if c.lower().endswith("_id") or c.lower().endswith("_ids")]
    seen = set()
    ordered = []
    for c in candidates:
        cl = c if isinstance(c, str) else str(c)
        if cl in seen:
            continue
        seen.add(cl); ordered.append(cl)

    n = len(var_df)
    ens_like = re.compile(r"^ENSG\d{6,}|^ENS[A-Z]{2,}\d{6,}", re.IGNORECASE)

    # pass 1: strictly Ensembl-like if possible
    for col in ordered:
        if col in var_df.columns:
            vals = pd.Series(var_df[col]).astype(str)
            if vals.isna().any():
                continue
            if vals.nunique(dropna=True) != n:
                continue
            if (vals.str.match(ens_like)).mean() > 0.8:  # mostly Ensembl-like
                return col
    # pass 2: any unique ID column
    for col in ordered:
        if col in var_df.columns:
            vals = pd.Series(var_df[col]).astype(str)
            if vals.isna().any():
                continue
            if vals.nunique(dropna=True) == n:
                return col
    return None

def _embed_cluster_reference(ad_ref, hvgs_mask, n_pcs=N_PCS, n_neighbors=N_NEI,
                             leiden_res=LEIDEN_RES, random_state=RANDOM_STATE):
    ad_ref = ad_ref.copy()
    ad_ref.var["highly_variable"] = hvgs_mask
    sc.pp.pca(ad_ref, n_comps=n_pcs, use_highly_variable=True, layer="lognorm",
              svd_solver="arpack", random_state=random_state)
    sc.pp.neighbors(ad_ref, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=random_state)
    # UMAP mainly for visualization; ingest later will project runs into this PCA
    try:
        if _USE_CUML:
            from cuml.manifold import UMAP as cuUMAP
            emb = cuUMAP(n_neighbors=n_neighbors, n_components=2, min_dist=0.5,
                         random_state=random_state).fit_transform(ad_ref.obsm["X_pca"])
            ad_ref.obsm["X_umap"] = np.asarray(emb)
        else:
            sc.tl.umap(ad_ref, min_dist=0.5, random_state=random_state)
    except Exception:
        sc.tl.umap(ad_ref, min_dist=0.5, random_state=random_state)
    sc.tl.leiden(ad_ref, flavor="igraph", resolution=leiden_res,
                 key_added="leiden_ref", random_state=random_state)
    return ad_ref

def _pick_reference_row(runs_df: pd.DataFrame) -> pd.Series:
    """
    Choose one file to serve as the (deepest) reference when replicates exist:
      1) Max target_rpc
      2) Among those, choose smallest 'rep' (deterministic)
      3) First row if multiple remain
    """
    max_rpc = runs_df["target_rpc"].astype(int).max()
    sub = runs_df[runs_df["target_rpc"].astype(int) == max_rpc].copy()
    sub = sub.sort_values(["rep"]).reset_index(drop=True)
    # Prefer an existing file
    for _, r in sub.iterrows():
        if os.path.exists(r["h5ad"]):
            return r
    # Fallback to first
    return sub.iloc[0]

# -------------------- LOAD SCHEMA & RUNS --------------------
schema = load_schema(SCHEMA_JSON)
runs_df = read_runs_index(schema)
print(f"[loaded] {len(runs_df)} subsample runs from index: {schema['runs_index_path']}")
print(runs_df.head())

# Choose reference as the largest target_rpc (deterministic replicate selection)
ref_row = _pick_reference_row(runs_df)
REF_H5AD = ref_row["h5ad"]
if not os.path.exists(REF_H5AD):
    raise FileNotFoundError(f"Reference h5ad not found: {REF_H5AD}")
print(f"[ref-pick] target_rpc={int(ref_row['target_rpc'])} | rep={int(ref_row.get('rep', 0))} | path={REF_H5AD}")

# -------------------- PREPARE REFERENCE --------------------
ad_ref = sc.read_h5ad(REF_H5AD)
print(f"[ref] {ad_ref.n_obs:,} cells × {ad_ref.n_vars:,} genes")

# Ensure log-normalized layer exists
_ensure_lognorm(ad_ref, layer="lognorm")

# Compute HVGs on the reference (single, fixed mask used for all runs)
hvg_mask = _compute_hvgs_on_ref(ad_ref, n_top=2000, layer="lognorm")

# Choose stable gene ID column (for later alignment in ref order)
gene_id_col = _select_gene_id_column(ad_ref.var)
if gene_id_col is None:
    print("[warn] No stable gene ID column found; will fall back to gene NAMES for alignment.")
else:
    print(f"[ref] Using '{gene_id_col}' as stable feature ID column for alignment.")

# Build reference PCA/NN/UMAP/Leiden ONCE (fixed space)
ad_ref = _embed_cluster_reference(ad_ref, hvg_mask,
                                  n_pcs=N_PCS, n_neighbors=N_NEI,
                                  leiden_res=LEIDEN_RES, random_state=RANDOM_STATE)

# Reference DE (string-typed groups)
ad_ref.obs["leiden_ref"] = ad_ref.obs["leiden_ref"].astype(str)
if "de_ref" not in ad_ref.uns:
    sc.tl.rank_genes_groups(ad_ref, groupby="leiden_ref", method="wilcoxon",
                            key_added="de_ref", use_raw=False, layer="lognorm", n_genes=300)

# Save prepared reference (with PCA/UMAP/Leiden/DE)
REF_OUT = os.path.join(ANALYSIS_ROOT, "reference_with_embedding.h5ad")
ad_ref.write_h5ad(REF_OUT)
print(f"[save] reference with clustering → {REF_OUT}")

# -------------------- Persist analysis schema for later cells --------------------
analysis_schema = {
    "inputs": {
        "artifact_schema_json": SCHEMA_JSON,
        "runs_index_csv": schema["runs_index_path"],
        "reference_h5ad": REF_OUT,
        "reads_col_pattern": schema.get("reads_col_pattern"),
        "umis_col_pattern": schema.get("umis_col_pattern"),
        "saturation_col_pattern": schema.get("saturation_col_pattern"),
        "gene_id_column": gene_id_col,     # may be None → fallback to var_names later
        "has_replicates": "rep" in runs_df.columns,
        "rep_column": "rep" if "rep" in runs_df.columns else None,
    },
    "params": {
        "n_pcs": N_PCS, "n_neighbors": N_NEI, "knn_overlap_k": KNN_OVERLAP_K,
        "leiden_res": LEIDEN_RES, "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS, "use_cuml_umap": bool(_USE_CUML),
        "do_car_analysis": bool(DO_CAR_ANALYSIS),
    },
    "outputs": {
        "clustering_metrics_csv": os.path.join(ANALYSIS_ROOT, "clustering_stability_metrics.csv"),
        "deg_metrics_csv":        os.path.join(ANALYSIS_ROOT, "deg_stability_metrics.csv"),
        "car_deg_traces_csv":     os.path.join(ANALYSIS_ROOT, "car_deg_traces.csv"),
        "figures": {
            "stability_panel_png": os.path.join(FIG_DIR, "stability_vs_reads.png"),
            "de_metrics_panel_png":os.path.join(FIG_DIR, "de_metrics_panel.png"),
            "car_deg_traces_png":  os.path.join(FIG_DIR, "car_deg_traces.png"),
            "umap_grid_png":       os.path.join(FIG_DIR, "umap_grid.png"),
        },
    },
}
ANALYSIS_SCHEMA_JSON = os.path.join(ANALYSIS_ROOT, "analysis_artifact_schema.json")
with open(ANALYSIS_SCHEMA_JSON, "w") as f:
    json.dump(analysis_schema, f, indent=2)
print(f"[save] analysis schema → {ANALYSIS_SCHEMA_JSON}")

gc.collect()
print("[done] Cell 1 ready: reference (replicate-aware) + analysis schema written.")


KeyError: 'path'

In [2]:
# === Cell 2 (replicate-aware): ingest-based metrics, save to H5ADs, DE robustness,
#     optional CAR+, UMAP grid, replicate averages with error bars,
#     and marker-gene mean transcript trajectories ===
import os, json, gc, warnings, math, re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse import issparse
from scipy.stats import spearmanr
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 2

# --------- locate the analysis schema produced in Cell 1 ----------
ANALYSIS_ROOT = "/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/analysis_rpc_clamped"
ASCH_JSON = os.path.join(ANALYSIS_ROOT, "analysis_artifact_schema.json")
assert os.path.exists(ASCH_JSON), f"Missing analysis schema: {ASCH_JSON}"

with open(ASCH_JSON, "r") as f:
    ASCH = json.load(f)

RUNS_CSV          = ASCH["inputs"]["runs_index_csv"]
REF_H5AD          = ASCH["inputs"]["reference_h5ad"]
READS_COL_PATTERN = ASCH["inputs"]["reads_col_pattern"]
UMIS_COL_PATTERN  = ASCH["inputs"]["umis_col_pattern"]
SAT_COL_PATTERN   = ASCH["inputs"]["saturation_col_pattern"]
GENE_ID_COL       = ASCH["inputs"]["gene_id_column"]  # may be None
HAS_REPS          = bool(ASCH["inputs"].get("has_replicates", False))
REP_COL           = ASCH["inputs"].get("rep_column", "rep") if HAS_REPS else None

OUT_CLUSTERING_CSV     = ASCH["outputs"]["clustering_metrics_csv"]
OUT_DEG_CSV            = ASCH["outputs"]["deg_metrics_csv"]
OUT_CARTRACES_FC_CSV   = ASCH["outputs"]["car_deg_traces_csv"]
OUT_CARTRACES_PADJ_CSV = os.path.join(ANALYSIS_ROOT, "car_deg_traces_padj.csv")
# new outputs for marker-gene trajectories
OUT_MARKER_MEAN_CSV    = os.path.join(ANALYSIS_ROOT, "marker_gene_mean_counts.csv")

FIG_STAB    = ASCH["outputs"]["figures"]["stability_panel_png"]
FIG_DE      = ASCH["outputs"]["figures"]["de_metrics_panel_png"]
FIG_CAR_FC  = ASCH["outputs"]["figures"]["car_deg_traces_png"]
FIG_CAR_P   = os.path.join(os.path.dirname(FIG_CAR_FC), "car_deg_traces_padj.png")
FIG_GRID    = ASCH["outputs"]["figures"]["umap_grid_png"]
FIG_MARKERS = os.path.join(os.path.dirname(FIG_STAB), "marker_gene_mean_counts.png")

P              = ASCH["params"]
N_PCS          = int(P.get("n_pcs", 50))
N_NEI          = int(P.get("n_neighbors", 20))
KNN_OVERLAP_K  = int(P.get("knn_overlap_k", 15))
LEIDEN_RES     = float(P.get("leiden_res", 0.6))
RANDOM_STATE   = int(P.get("random_state", 0))
USE_CUML       = bool(P.get("use_cuml_umap", False))
DO_CAR_ANALYSIS= bool(P.get("do_car_analysis", True))

for pth in [OUT_CLUSTERING_CSV, OUT_DEG_CSV, OUT_CARTRACES_FC_CSV, OUT_CARTRACES_PADJ_CSV,
            OUT_MARKER_MEAN_CSV,
            FIG_STAB, FIG_DE, FIG_CAR_FC, FIG_CAR_P, FIG_GRID, FIG_MARKERS]:
    os.makedirs(os.path.dirname(pth), exist_ok=True)

# ---------------- helpers ----------------
def _ensure_lognorm(ad, layer="lognorm", target_sum=1e4):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=target_sum, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _save_basic_qc_to_obs(ad):
    X = ad.X
    if issparse(X):
        n_counts = np.asarray(X.sum(axis=1)).ravel()
        n_genes  = np.asarray((X > 0).sum(axis=1)).ravel()
    else:
        n_counts = X.sum(axis=1).ravel()
        n_genes  = (X > 0).sum(axis=1).ravel()
    ad.obs["total_counts"] = n_counts
    ad.obs["n_genes_by_counts"] = n_genes

def _build_ref_id_index(ad_ref, gene_id_col: str | None):
    """Series length = n_vars (ref), values = stable IDs (fallback to var_names)."""
    if gene_id_col and gene_id_col in ad_ref.var.columns:
        vals = pd.Series(ad_ref.var[gene_id_col]).astype(str)
        if (vals.isna().any()) or (vals.nunique() != len(vals)):
            gene_id_col = None
    if not gene_id_col:
        vals = pd.Series(ad_ref.var_names).astype(str)
    return vals.reset_index(drop=True)

def _align_by_gene_ids(ad_q, ad_ref, gene_id_col: str | None):
    """
    Align query to reference variable order using stable IDs (prefer Ensembl-like).
    Returns (ref_sub, q_sub, ref_mask_bool, q_idx_in_q) with *intersection* only.
    """
    ref_ids = _build_ref_id_index(ad_ref, gene_id_col)
    if gene_id_col and gene_id_col in ad_q.var.columns:
        q_ids = pd.Series(ad_q.var[gene_id_col]).astype(str).reset_index(drop=True)
    else:
        q_ids = pd.Series(ad_q.var_names).astype(str).reset_index(drop=True)

    q_lut = pd.Series(q_ids.index.values, index=q_ids.values)
    take = []
    for rid in ref_ids:
        if rid in q_lut.index:
            take.append(int(q_lut.loc[rid]))
        else:
            take.append(None)

    mask = np.array([j is not None for j in take], dtype=bool)
    if not mask.any():
        raise RuntimeError("No overlapping genes between run and reference after ID alignment.")
    q_idx = np.array([j for j in take if j is not None], dtype=int)
    ref_sub = ad_ref[:, mask].copy()
    q_sub   = ad_q[:, q_idx].copy()
    if "highly_variable" in ad_ref.var.columns:
        ref_hvg = ad_ref.var["highly_variable"].values[mask]
        q_sub.var["highly_variable"] = ref_hvg
    return ref_sub, q_sub, mask, q_idx

def _project_into_ref_space(ad_q, ad_ref, n_neighbors=N_NEI, n_pcs=N_PCS, leiden_res=LEIDEN_RES):
    """
    Project ad_q into ad_ref PCA space via scanpy.ingest on *float lognorm*.
    Then rebuild neighbors on X_pca, compute UMAP & Leiden (key: 'leiden_ds').
    """
    X_backup = ad_q.X
    try:
        X_ln = ad_q.layers["lognorm"]
        if sparse.issparse(X_ln): X_ln = X_ln.astype(np.float32)
        else: X_ln = X_ln.astype(np.float32, copy=False)
        ad_q.X = X_ln
        sc.tl.ingest(ad_q, ad_ref, obs=None)  # writes X_pca etc.
    finally:
        ad_q.X = X_backup

    sc.pp.neighbors(ad_q, n_neighbors=n_neighbors, n_pcs=n_pcs, use_rep="X_pca", random_state=RANDOM_STATE)
    try:
        if USE_CUML:
            from cuml.manifold import UMAP as cuUMAP
            emb = cuUMAP(n_neighbors=n_neighbors, n_components=2, min_dist=0.5,
                         random_state=RANDOM_STATE).fit_transform(ad_q.obsm["X_pca"])
            ad_q.obsm["X_umap"] = np.asarray(emb)
        else:
            sc.tl.umap(ad_q, min_dist=0.5, random_state=RANDOM_STATE)
    except Exception:
        sc.tl.umap(ad_q, min_dist=0.5, random_state=RANDOM_STATE)

    sc.tl.leiden(ad_q, flavor="igraph", resolution=leiden_res, key_added="leiden_ds", random_state=RANDOM_STATE)
    ad_q.obs["leiden"] = ad_q.obs["leiden_ds"].astype(str).values  # convenience alias
    return ad_q

def _neighbor_overlap_jaccard(A_ref, A_q, k=15) -> float:
    """Average Jaccard of top-k neighbors per cell between two CSR graphs with same cell order."""
    if not sparse.isspmatrix_csr(A_ref): A_ref = A_ref.tocsr()
    if not sparse.isspmatrix_csr(A_q):   A_q   = A_q.tocsr()
    n = A_ref.shape[0]
    tot = 0.0; cnt = 0
    for i in range(n):
        sr = slice(A_ref.indptr[i], A_ref.indptr[i+1])
        idx_r = A_ref.indices[sr]; dat_r = A_ref.data[sr]
        if idx_r.size == 0: continue
        kr = min(k, idx_r.size)
        top_r = idx_r[np.argpartition(-dat_r, kr-1)[:kr]]

        sd = slice(A_q.indptr[i], A_q.indptr[i+1])
        idx_d = A_q.indices[sd]; dat_d = A_q.data[sd]
        if idx_d.size == 0: continue
        kd = min(k, idx_d.size)
        top_d = idx_d[np.argpartition(-dat_d, kd-1)[:kd]]

        s1, s2 = set(top_r), set(top_d)
        u = len(s1 | s2)
        if u:
            tot += len(s1 & s2)/u; cnt += 1
    return float(tot/cnt) if cnt else np.nan

def _hungarian_match(y_true, y_pred):
    tcat = pd.Categorical(y_true); pcat = pd.Categorical(y_pred)
    cm = np.zeros((len(tcat.categories), len(pcat.categories)), dtype=int)
    for t, p in zip(tcat.codes, pcat.codes):
        if t >= 0 and p >= 0: cm[t, p] += 1
    r, c = linear_sum_assignment(-cm)
    mapping = {pcat.categories[cj]: tcat.categories[ri] for ri, cj in zip(r, c)}
    acc = cm[r, c].sum() / cm.sum() if cm.sum() > 0 else np.nan
    return mapping, float(acc)

def _deg_table(ad, key, group):
    try:
        return sc.get.rank_genes_groups_df(ad, key=key, group=group)
    except Exception:
        rg = ad.uns[key]
        names = pd.Series(rg["names"][group], name="names")
        pvals_adj = pd.Series(rg["pvals_adj"][group], name="pvals_adj")
        logfc = pd.Series(rg["logfoldchanges"][group], name="logfoldchanges")
        return pd.DataFrame({"names": names, "pvals_adj": pvals_adj, "logfoldchanges": logfc})

def _topN_jaccard(list_a, list_b, N=20):
    if len(list_a)==0 and len(list_b)==0: return np.nan
    a = set(list_a[:N]); b = set(list_b[:N])
    u = len(a|b)
    return len(a&b)/u if u else np.nan

def _total_reads_for_run(ad, target_rpc):
    if READS_COL_PATTERN:
        col = READS_COL_PATTERN.format(target_rpc=int(target_rpc))
        if col in ad.obs.columns:
            return float(np.asarray(ad.obs[col]).sum())
    cand = [c for c in ad.obs.columns if str(c).startswith("reads_sim_")]
    if cand:
        return float(np.asarray(ad.obs[cand[0]]).sum())
    return float(np.asarray(ad.X.sum(axis=1)).sum())

def _find_gene_idx(ad, gene_symbol: str):
    # Try var_names
    if gene_symbol in ad.var_names:
        return ad.var_names.get_loc(gene_symbol)
    up = ad.var_names.str.upper() == gene_symbol.upper()
    if up.any():
        return int(np.where(up)[0][0])
    # Try common symbol columns
    for col in ["gene_symbol","gene_symbols","symbol","name","feature_name"]:
        if col in ad.var.columns:
            ser = pd.Series(ad.var[col]).astype(str)
            hit = ser.str.upper() == gene_symbol.upper()
            if hit.any():
                return int(np.where(hit)[0][0])
    # partial contains (last resort)
    mask = ad.var_names.str.upper().str.contains(gene_symbol.upper())
    if mask.any():
        return int(np.where(mask)[0][0])
    return None

# ---------------- load reference & runs ----------------
runs = pd.read_csv(RUNS_CSV).reset_index(drop=True)
if REP_COL is None or REP_COL not in runs.columns:
    runs["rep"] = 0
    REP_COL = "rep"
runs = runs.sort_values(["target_rpc", REP_COL]).reset_index(drop=True)

ad_ref = sc.read_h5ad(REF_H5AD)
_ensure_lognorm(ad_ref, layer="lognorm")

# metrics collectors
metrics_rows, deg_rows = [], []
marker_rows = []  # for mean transcript count per cell

MARKER_GENES = ["CD34","CD3E","CD4","CD68","CD8A","COL5A2","EPCAM","MS4A1","MZB1"]

print(f"[ref] {ad_ref.n_obs:,} cells × {ad_ref.n_vars:,} genes | GENE_ID_COL={GENE_ID_COL}")
print("[start] per-run projection into reference space via ingest")
for _, r in runs.iterrows():
    rpc = int(r["target_rpc"])
    rep = int(r.get(REP_COL, 0))
    h5ad_path = str(r["h5ad"])
    if not os.path.exists(h5ad_path):
        print(f"[skip] missing {h5ad_path}")
        continue

    print(f"\n[run] target_rpc={rpc} | rep={rep} → {os.path.basename(h5ad_path)}")
    ad = sc.read_h5ad(h5ad_path)
    _ensure_lognorm(ad, layer="lognorm")
    _save_basic_qc_to_obs(ad)

    # --- align genes by stable IDs to reference order (intersection only) ---
    ref_al, ad_al, ref_mask, ad_idx = _align_by_gene_ids(ad, ad_ref, GENE_ID_COL)

    # --- project (ingest) into reference PCA space; then neighbors/UMAP/Leiden ---
    ad_al = _project_into_ref_space(ad_al, ad_ref, n_neighbors=N_NEI, n_pcs=N_PCS, leiden_res=LEIDEN_RES)

    # ---- DE on aligned object (for fair comparison to ref DE) ----
    sc.tl.rank_genes_groups(ad_al, groupby="leiden_ds", method="wilcoxon",
                            key_added="de_aligned", use_raw=False, layer="lognorm", n_genes=300)

    # --- Clustering comparatives (same cells/order) ---
    y_true = ref_al.obs["leiden_ref"].astype(str).values
    y_pred = ad_al.obs["leiden_ds"].astype(str).values
    ARI = adjusted_rand_score(y_true, y_pred)
    NMI = normalized_mutual_info_score(y_true, y_pred)
    mapping, map_acc = _hungarian_match(y_true, y_pred)
    nnJ = _neighbor_overlap_jaccard(ref_al.obsp["connectivities"], ad_al.obsp["connectivities"], k=KNN_OVERLAP_K)

    # --- DE robustness (mean across matched clusters) ---
    jcats, rhos = [], []
    for qlab, tlab in mapping.items():
        df_ref = _deg_table(ref_al, key="de_ref", group=tlab)
        df_q   = _deg_table(ad_al,  key="de_aligned", group=qlab)
        jcats.append(_topN_jaccard(
            df_ref.sort_values("pvals_adj")["names"].tolist(),
            df_q.sort_values("pvals_adj")["names"].tolist(), N=20
        ))
        m = df_ref[["names","logfoldchanges"]].merge(
            df_q[["names","logfoldchanges"]], on="names",
            suffixes=("_ref","_q"))
        if len(m) >= 10:
            rhos.append(spearmanr(m["logfoldchanges_ref"], m["logfoldchanges_q"]).correlation)

        # per-cluster drill-down
        deg_rows.append(dict(
            target_rpc=rpc, rep=rep, cluster_ref=str(tlab), cluster_q=str(qlab),
            top20_jaccard=_topN_jaccard(
                df_ref.sort_values("pvals_adj")["names"].tolist(),
                df_q.sort_values("pvals_adj")["names"].tolist(), N=20
            ),
            h5ad=h5ad_path
        ))

    de_top20 = float(np.nanmean(jcats)) if jcats else np.nan
    de_rho   = float(np.nanmean(rhos))  if rhos else np.nan

    # --- totals & meta ---
    total_reads = _total_reads_for_run(ad, rpc)
    reads_m = total_reads / 1e6
    metrics_rows.append(dict(
        target_rpc=rpc, rep=rep, total_reads=total_reads, reads_millions=reads_m,
        ARI=ARI, NMI=NMI, cluster_map_acc=map_acc, neighbor_jaccard=nnJ,
        de_top20_jaccard=de_top20, de_logfc_spearman=de_rho,
        h5ad=h5ad_path
    ))

    # --- marker genes: mean transcript count per cell (raw UMIs in X) ---
    X = ad.X.toarray() if issparse(ad.X) else np.asarray(ad.X)
    for g in MARKER_GENES:
        j = _find_gene_idx(ad, g)
        mean_ct = float(np.mean(X[:, j])) if j is not None else np.nan
        marker_rows.append(dict(target_rpc=rpc, rep=rep, reads_millions=reads_m, gene=g, mean_transcripts_per_cell=mean_ct))

    # --- persist aligned embeddings/graph/labels back into ORIGINAL H5AD ---
    ad.obsm["X_pca_refspace"] = ad_al.obsm["X_pca"]
    if "X_umap" in ad_al.obsm: ad.obsm["X_umap_refspace"] = ad_al.obsm["X_umap"]
    ad.obsp["connectivities_refspace"] = ad_al.obsp["connectivities"]
    if "distances" in ad_al.obsp: ad.obsp["distances_refspace"] = ad_al.obsp["distances"]
    ad.obs["leiden_ds"] = pd.Categorical(ad_al.obs["leiden_ds"].astype(str).reindex(ad.obs_names).fillna("NA"))
    ad.obs["leiden"]    = ad.obs["leiden_ds"].astype(str)

    if "total_counts" not in ad.obs: _save_basic_qc_to_obs(ad)

    ad.uns.setdefault("subsample_metrics", {})
    ad.uns["subsample_metrics"].update(dict(
        target_rpc=rpc, rep=rep, reads_millions=reads_m,
        ARI=ARI, NMI=NMI, cluster_map_acc=map_acc, neighbor_jaccard=nnJ,
        de_top20_jaccard=de_top20, de_logfc_spearman=de_rho,
        params=dict(n_pcs=N_PCS, n_neighbors=N_NEI, knn_overlap_k=KNN_OVERLAP_K,
                    leiden_res=LEIDEN_RES, random_state=RANDOM_STATE, use_cuml_umap=USE_CUML)
    ))

    try:
        ad.uns["de_aligned"] = ad_al.uns["de_aligned"]
    except Exception:
        pass

    ad.write_h5ad(h5ad_path)
    print(f"[save] updated embeddings/labels/metrics → {h5ad_path}")

    del ad, ad_al, ref_al
    gc.collect()

# ---------- Save per-run CSVs ----------
metrics_df = pd.DataFrame(metrics_rows).sort_values(["target_rpc","rep"])
deg_df     = pd.DataFrame(deg_rows).sort_values(["target_rpc","rep","cluster_ref"])
markers_df = pd.DataFrame(marker_rows).sort_values(["gene","target_rpc","rep"])
metrics_df.to_csv(OUT_CLUSTERING_CSV, index=False)
deg_df.to_csv(OUT_DEG_CSV, index=False)
markers_df.to_csv(OUT_MARKER_MEAN_CSV, index=False)
print(f"\n[save] clustering metrics → {OUT_CLUSTERING_CSV}")
print(f"[save] per-cluster DE metrics → {OUT_DEG_CSV}")
print(f"[save] marker-gene means → {OUT_MARKER_MEAN_CSV}")

# ---------- Aggregate over replicates (mean ± SD) ----------
def _agg_with_err(df, ycols, xcol="reads_millions"):
    """
    Returns: dict of DataFrames {metric: agg_df} where agg_df has columns:
      target_rpc, x_mean, x_sd, y_mean, y_sd, n
    """
    out = {}
    g = df.groupby("target_rpc", as_index=False)
    x_mean = g[xcol].mean().rename(columns={xcol:"x_mean"})
    x_sd   = g[xcol].std(ddof=1).rename(columns={xcol:"x_sd"})
    n      = g.size().rename(columns={"size":"n"})
    base = x_mean.merge(x_sd, on="target_rpc").merge(n, on="target_rpc")
    for y in ycols:
        y_m = g[y].mean().rename(columns={y:"y_mean"})
        y_s = g[y].std(ddof=1).rename(columns={y:"y_sd"})
        out[y] = base.merge(y_m, on="target_rpc").merge(y_s, on="target_rpc").sort_values("x_mean")
    return out

# metrics aggregation
metric_names = ["ARI","NMI","cluster_map_acc","neighbor_jaccard","de_top20_jaccard","de_logfc_spearman"]
agg = _agg_with_err(metrics_df, metric_names, xcol="reads_millions")

# marker genes aggregation
markers_agg = {}
for gname, sub in markers_df.groupby("gene"):
    ag = sub.groupby("target_rpc", as_index=False).agg(
        x_mean=("reads_millions","mean"),
        x_sd=("reads_millions","std"),
        y_mean=("mean_transcripts_per_cell","mean"),
        y_sd=("mean_transcripts_per_cell","std"),
        n=("mean_transcripts_per_cell","size"),
    ).sort_values("x_mean")
    markers_agg[gname] = ag

# ---------------- Figures: clustering & DE panels with error bars ----------------
if not metrics_df.empty:
    # Stability panel: show per-run points (light) + mean±SD error bars
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), constrained_layout=True)
    axes = axes.ravel()
    titles = [
        "ARI vs total reads",
        "NMI vs total reads",
        "Cluster map accuracy (Hungarian)",
        f"kNN Jaccard (k={KNN_OVERLAP_K})",
        "DE Top-20 Jaccard",
        "DE log2FC Spearman",
    ]
    for idx, m in enumerate(metric_names):
        ax = axes[idx]
        # scatter all reps
        ax.scatter(metrics_df["reads_millions"], metrics_df[m], s=15, alpha=0.4)
        # mean ± SD bars
        A = agg[m]
        ax.errorbar(A["x_mean"], A["y_mean"], yerr=A["y_sd"], fmt='-o', capsize=3)
        ax.set_title(titles[idx])
        ax.set_xlabel("Reads (M)")
        ax.grid(True, alpha=0.3)
    plt.savefig(FIG_STAB, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] stability panel (mean±SD over reps) → {FIG_STAB}")

    # DE-only compact panel
    plt.figure(figsize=(10, 4))
    # Jaccard
    A = agg["de_top20_jaccard"]
    plt.subplot(1,2,1)
    plt.scatter(metrics_df["reads_millions"], metrics_df["de_top20_jaccard"], s=15, alpha=0.4)
    plt.errorbar(A["x_mean"], A["y_mean"], yerr=A["y_sd"], fmt='-o', capsize=3)
    plt.title("DE Top-20 Jaccard"); plt.xlabel("Reads (M)"); plt.grid(True, alpha=0.3)
    # Spearman
    A = agg["de_logfc_spearman"]
    plt.subplot(1,2,2)
    plt.scatter(metrics_df["reads_millions"], metrics_df["de_logfc_spearman"], s=15, alpha=0.4)
    plt.errorbar(A["x_mean"], A["y_mean"], yerr=A["y_sd"], fmt='-o', capsize=3)
    plt.title("DE log2FC Spearman"); plt.xlabel("Reads (M)"); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DE, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] DE metrics panel (mean±SD) → {FIG_DE}")

# ---------------- Optional: CAR+ vs CAR- trajectories (unchanged aggregation; per-run sweep) ----------------
if DO_CAR_ANALYSIS:
    CAR_GENE = "ciltacel"  # used to define CAR+ cells; we exclude it from plotted/top genes
    if not metrics_df.empty:
        runs_df = pd.read_csv(RUNS_CSV).sort_values(["target_rpc", REP_COL])
        # pick deepest by average reads (then smallest rep deterministically)
        depth_means = metrics_df.groupby("target_rpc", as_index=False)["reads_millions"].mean()
        rpc_max = int(depth_means.iloc[depth_means["reads_millions"].argmax()]["target_rpc"])
        h5_max_rows = runs_df[runs_df["target_rpc"].astype(int)==rpc_max].sort_values(REP_COL)
        h5_max = str(h5_max_rows.iloc[0]["h5ad"])

        ad_max = sc.read_h5ad(h5_max)
        _ensure_lognorm(ad_max, layer="lognorm")

        def _find_gene_index(ad, gene_name):
            if gene_name in ad.var_names: return ad.var_names.get_loc(gene_name)
            up = ad.var_names.str.upper() == gene_name.upper()
            if up.any(): return int(np.where(up)[0][0])
            return None

        j_car = _find_gene_index(ad_max, CAR_GENE)
        if j_car is not None:
            Xr = ad_max.X.toarray() if issparse(ad_max.X) else np.asarray(ad_max.X)
            car_pos_mask = Xr[:, j_car] > 0

            ad_max_tmp = ad_max.copy()
            ad_max_tmp.obs["__bin__"] = np.where(car_pos_mask, "CARpos", "CARneg")
            sc.tl.rank_genes_groups(ad_max_tmp, groupby="__bin__", groups=["CARpos"], reference="CARneg",
                                    method="wilcoxon", use_raw=False, layer="lognorm",
                                    key_added="de_carmax", n_genes=1000)
            df_max = sc.get.rank_genes_groups_df(ad_max_tmp, key="de_carmax", group="CARpos").sort_values("pvals_adj")
            TOP10 = [g for g in df_max["names"].tolist() if g != CAR_GENE][:10]
            print(f"[CAR] top10 (excluding {CAR_GENE}): {TOP10}")

            def _log2fc_for_genes(ad, pos_mask, genes, layer="lognorm", eps=1e-9):
                X = ad.layers[layer] if layer in ad.layers else ad.X
                X = X.toarray() if issparse(X) else np.asarray(X)
                pos = X[pos_mask]; neg = X[~pos_mask]
                gidx = [ad.var_names.get_loc(g) if g in ad.var_names else None for g in genes]
                out = []
                for g, j in zip(genes, gidx):
                    if j is None: out.append(np.nan); continue
                    mu_pos = float(np.mean(pos[:, j])); mu_neg = float(np.mean(neg[:, j]))
                    out.append(np.log2((mu_pos + eps) / (mu_neg + eps)))
                return out

            fc_rows, padj_rows = [], []
            for _, rr in runs_df.iterrows():
                rpc = int(rr["target_rpc"]); h5ad = str(rr["h5ad"])
                if not os.path.exists(h5ad): continue
                ad = sc.read_h5ad(h5ad)
                _ensure_lognorm(ad, layer="lognorm")

                j_local = _find_gene_index(ad, CAR_GENE)
                if j_local is None: 
                    continue
                Xl = ad.X.toarray() if issparse(ad.X) else np.asarray(ad.X)
                car_mask = Xl[:, j_local] > 0

                ad_tmp = ad.copy()
                ad_tmp.obs["__bin__"] = np.where(car_mask, "CARpos", "CARneg")
                sc.tl.rank_genes_groups(ad_tmp, groupby="__bin__", groups=["CARpos"], reference="CARneg",
                                        method="wilcoxon", use_raw=False, layer="lognorm",
                                        key_added="de_car", n_genes=1000)
                df_run = sc.get.rank_genes_groups_df(ad_tmp, key="de_car", group="CARpos")

                total_reads = _total_reads_for_run(ad, rpc)
                reads_m = total_reads / 1e6

                log2fcs = _log2fc_for_genes(ad, car_mask, TOP10, layer="lognorm")
                for g, l2 in zip(TOP10, log2fcs):
                    fc_rows.append(dict(target_rpc=rpc, reads_millions=reads_m, gene=g, log2FC=l2))

                padj_map = dict(zip(df_run["names"], df_run["pvals_adj"]))
                for g in TOP10:
                    padj = float(padj_map.get(g, np.nan))
                    neglog10 = -np.log10(padj) if (padj is not None and padj > 0) else np.nan
                    padj_rows.append(dict(target_rpc=rpc, reads_millions=reads_m, gene=g,
                                          padj=padj, neglog10_padj=neglog10))

                del ad_tmp, ad
                gc.collect()

            traces_fc_df   = pd.DataFrame(fc_rows).sort_values(["gene","reads_millions"])
            traces_padj_df = pd.DataFrame(padj_rows).sort_values(["gene","reads_millions"])
            traces_fc_df.to_csv(OUT_CARTRACES_FC_CSV, index=False)
            traces_padj_df.to_csv(OUT_CARTRACES_PADJ_CSV, index=False)
            print(f"[save] CAR+ log2FC trajectories → {OUT_CARTRACES_FC_CSV}")
            print(f"[save] CAR+ padj trajectories → {OUT_CARTRACES_PADJ_CSV}")

            # plots (per-run scatter + mean line if multiple values at same RPC)
            plt.figure(figsize=(12, 7))
            for gname, df_g in traces_fc_df.groupby("gene"):
                plt.scatter(df_g["reads_millions"], df_g["log2FC"], s=15, alpha=0.5, label=None)
                g_agg = df_g.groupby("reads_millions", as_index=False)["log2FC"].mean()
                plt.plot(g_agg["reads_millions"], g_agg["log2FC"], marker="o", label=gname)
            plt.axhline(0, linestyle="--", linewidth=1)
            plt.xlabel("Total reads (millions)"); plt.ylabel("log2FC (CAR+ vs CAR−), lognorm")
            plt.title("Top-10 CAR⁺ DE genes (excl. ciltacel): log2FC vs reads")
            plt.legend(ncol=2, fontsize=8); plt.grid(True, alpha=0.3)
            plt.savefig(FIG_CAR_FC, dpi=160, bbox_inches="tight"); plt.close()
            print(f"[save] CAR log2FC traces → {FIG_CAR_FC}")

            plt.figure(figsize=(12, 7))
            for gname, df_g in traces_padj_df.groupby("gene"):
                plt.scatter(df_g["reads_millions"], df_g["neglog10_padj"], s=15, alpha=0.5, label=None)
                g_agg = df_g.groupby("reads_millions", as_index=False)["neglog10_padj"].mean()
                plt.plot(g_agg["reads_millions"], g_agg["neglog10_padj"], marker="o", label=gname)
            plt.xlabel("Total reads (millions)"); plt.ylabel("-log10 adjusted p-value")
            plt.title("Top-10 CAR⁺ DE genes (excl. ciltacel): significance vs reads")
            plt.legend(ncol=2, fontsize=8); plt.grid(True, alpha=0.3)
            plt.savefig(FIG_CAR_P, dpi=160, bbox_inches="tight"); plt.close()
            print(f"[save] CAR padj traces → {FIG_CAR_P}")
        else:
            print(f"[CAR][warn] Gene '{CAR_GENE}' not found in deepest run; skipping CAR analysis.")
    else:
        print("[CAR] metrics_df empty; skipping CAR analysis.")
else:
    print("[CAR] DO_CAR_ANALYSIS=False → skipping CAR analysis.")

# ---------------- Marker-gene mean transcript trajectories (mean±SD across reps) ----------------
if not markers_df.empty:
    plt.figure(figsize=(12, 7))
    for gname, A in markers_agg.items():
        # per-run points
        df_g = markers_df[markers_df["gene"] == gname]
        plt.scatter(df_g["reads_millions"], df_g["mean_transcripts_per_cell"], s=12, alpha=0.35, label=None)
        # mean ± SD
        plt.errorbar(A["x_mean"], A["y_mean"], yerr=A["y_sd"], fmt='-o', capsize=3, label=gname)
    plt.xlabel("Total reads (millions)")
    plt.ylabel("Mean transcripts per cell")
    plt.title("Marker-gene mean transcripts per cell vs total reads (mean±SD across replicates)")
    plt.legend(ncol=3, fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.savefig(FIG_MARKERS, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] marker gene trajectories → {FIG_MARKERS}")

# ---------------- UMAP grid over all runs (clustered in refspace) ----------------
paths = runs.sort_values(["target_rpc", REP_COL])["h5ad"].tolist()
n = len(paths)
if n > 0:
    ncols = int(math.ceil(math.sqrt(n)))
    nrows = int(math.ceil(n / ncols))

    mins = np.array([np.inf, np.inf]); maxs = np.array([-np.inf, -np.inf])
    cache = []
    for p in paths:
        if not os.path.exists(p):
            cache.append((None, None, None)); continue
        ad = sc.read_h5ad(p)
        if "X_umap_refspace" in ad.obsm_keys():
            Xumap = np.asarray(ad.obsm["X_umap_refspace"])
        elif "X_umap" in ad.obsm_keys():
            Xumap = np.asarray(ad.obsm["X_umap"])
        else:
            _ensure_lognorm(ad, "lognorm")
            sc.pp.pca(ad, n_comps=20, layer="lognorm")
            sc.pp.neighbors(ad, n_neighbors=15, n_pcs=20)
            sc.tl.umap(ad, random_state=RANDOM_STATE)
            Xumap = np.asarray(ad.obsm["X_umap"])
            ad.write_h5ad(p)  # cache
        labs = ad.obs["leiden_ds"].astype(str).values if "leiden_ds" in ad.obs else (
               ad.obs["leiden"].astype(str).values if "leiden" in ad.obs else None)
        cache.append((Xumap, labs, os.path.basename(p)))
        mins = np.minimum(mins, Xumap.min(axis=0)); maxs = np.maximum(maxs, Xumap.max(axis=0))
        del ad; gc.collect()

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.8*ncols, 3.8*nrows), squeeze=False)
    axlist = axes.ravel()
    for i, (Xumap, labs, title) in enumerate(cache):
        ax = axlist[i]
        if Xumap is None: ax.axis("off"); continue
        if labs is None:
            ax.scatter(Xumap[:,0], Xumap[:,1], s=1, alpha=0.7)
        else:
            for lab in np.unique(labs):
                m = (labs == lab)
                ax.scatter(Xumap[m,0], Xumap[m,1], s=1, alpha=0.7)
        ax.set_title(title, fontsize=9)
        ax.set_xlim(mins[0], maxs[0]); ax.set_ylim(mins[1], maxs[1])
        ax.set_xticks([]); ax.set_yticks([])
    for j in range(len(cache), len(axlist)):
        axlist[j].axis("off")
    plt.tight_layout()
    plt.savefig(FIG_GRID, dpi=160, bbox_inches="tight"); plt.close()
    print(f"[save] UMAP grid → {FIG_GRID}")

print("[done] Cell 2 complete (replicate-aware metrics, error bars, marker-gene trajectories).")


KeyError: 'inputs'

NameError: name 'ad' is not defined

In [19]:
duo_data= sc.read_h5ad("/homes/olymp/julius.guntrum/tmp/Koeln/duo/3/tenx_like_subsampling_2/runs_rpc_clamped/subsample_MAPPED_rpc930_r0.h5ad")

In [21]:
duo_data.obs

,location_id,region,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,...,log1p_total_counts_ribo,pct_counts_ribo,annot_1,annot_2,annot_3,reads_sim_MAPPED_rpc930,umis_sim_MAPPED_rpc930,saturation_sim_MAPPED_rpc930,leiden_ds,leiden
cellid_000000001-1,0,cell_segmentation,134,5.153292,188,5.575949,53.612167,72.623574,100.000000,100.000000,...,0.0,0.0,Mesenchymal,Secretory,Mesoderm_2,401,188,0.531172,0,0
cellid_000000002-1,1,cell_segmentation,635,6.745236,755,6.953684,16.539197,26.099426,37.954111,66.634799,...,0.0,0.0,Mesenchymal,Fibroblast,Crypt_fibroblast_PI16,1491,755,0.493628,1,1
cellid_000000003-1,2,cell_segmentation,156,5.332719,195,5.594711,41.791045,60.447761,97.761194,100.000000,...,0.0,0.0,Mesenchymal,Secretory,Mesoderm_2,387,195,0.496124,2,2
cellid_000000004-1,3,cell_segmentation,420,6.347389,457,6.459905,18.495298,26.332288,42.006270,89.028213,...,0.0,0.0,Epithelial,Absorptive,Enterocyte,887,457,0.484780,3,3
cellid_000000005-1,4,cell_segmentation,92,4.770685,95,4.812184,45.081967,86.065574,100.000000,100.000000,...,0.0,0.0,Mesenchymal,Mesoderm,Mesoderm_2,216,95,0.560185,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cellid_000058243-1,58238,cell_segmentation,145,5.318120,186,5.587249,42.481203,61.278195,98.872180,100.000000,...,0.0,0.0,Epithelial,Absorptive,Enterocyte,382,186,0.513089,8,8
cellid_000058244-1,58239,cell_segmentation,727,6.825460,1243,7.444833,39.415205,47.192982,57.894737,75.438596,...,0.0,0.0,Epithelial,Absorptive,Enterocyte,2535,1243,0.509665,8,8
cellid_000058245-1,58240,cell_segmentation,184,5.529429,277,5.981414,49.113924,61.772152,87.088608,100.000000,...,0.0,0.0,Epithelial,Absorptive,Enterocyte,574,277,0.517422,8,8
cellid_000058246-1,58241,cell_segmentation,0,0.000000,0,0.000000,NaN,NaN,NaN,NaN,...,0.0,NaN,B and B plasma,Absorptive,Angiogenic_pericyte,0,0,NaN,11,11


In [5]:
from scipy.stats import spearmanr  # NEW: for DE log2FC Spearman
# ---------- DE helpers (NEW) ----------
def _deg_table(ad: sc.AnnData, key: str, group: str) -> pd.DataFrame:
    """Robustly pull an RG result table."""
    try:
        return sc.get.rank_genes_groups_df(ad, key=key, group=group)
    except Exception:
        rg = ad.uns[key]
        names = pd.Series(rg["names"][group], name="names")
        pvals_adj = pd.Series(rg["pvals_adj"][group], name="pvals_adj")
        logfc = pd.Series(rg["logfoldchanges"][group], name="logfoldchanges")
        return pd.DataFrame({"names": names, "pvals_adj": pvals_adj, "logfoldchanges": logfc})

def _topN_jaccard(list_a, list_b, N=20):
    if len(list_a)==0 and len(list_b)==0: return np.nan
    a = set(list_a[:N]); b = set(list_b[:N])
    u = len(a|b)
    return len(a&b)/u if u else np.nan

def _ensure_de(ad: sc.AnnData, groupby: str, key_added: str, n_genes: int = 300):
    """Compute rank_genes_groups if missing under key_added on this object."""
    if key_added not in ad.uns:
        sc.tl.rank_genes_groups(ad, groupby=groupby, method="wilcoxon",
                                use_raw=False, layer="lognorm",
                                key_added=key_added, n_genes=n_genes)

def _compute_de_metrics(
    ad_ref_sub: sc.AnnData,
    ad_run_sub: sc.AnnData,
    ref_group_key: str,
    run_group_key: str,
    mapping_true_to_pred: dict[str, str],
    topN: int = 20
) -> tuple[float, float]:
    """
    Returns (DE Top-20 Jaccard mean across matched clusters, DE log2FC Spearman mean).
    mapping_true_to_pred is from Hungarian match on (ref labels → run labels).
    """
    # Compute DE tables (on the *same* cells that were used for metrics)
    _ensure_de(ad_ref_sub, groupby=ref_group_key, key_added="__de_ref_tmp")
    _ensure_de(ad_run_sub, groupby=run_group_key, key_added="__de_run_tmp")

    # invert mapping: run_label -> ref_label
    mapping_pred_to_true = {v: k for k, v in mapping_true_to_pred.items()}

    jcats, rhos = [], []
    for qlab, tlab in mapping_pred_to_true.items():
        # pull DE tables
        try:
            df_ref = _deg_table(ad_ref_sub, "__de_ref_tmp", str(tlab)).sort_values("pvals_adj")
            df_run = _deg_table(ad_run_sub, "__de_run_tmp", str(qlab)).sort_values("pvals_adj")
        except Exception:
            continue

        # Top-20 Jaccard by adjusted p-value
        jcats.append(_topN_jaccard(df_ref["names"].tolist(), df_run["names"].tolist(), N=topN))

        # Spearman on log2FC over overlapping genes
        m = df_ref[["names","logfoldchanges"]].merge(
            df_run[["names","logfoldchanges"]], on="names",
            suffixes=("_ref","_run")
        )
        if len(m) >= 10:
            rhos.append(spearmanr(m["logfoldchanges_ref"], m["logfoldchanges_run"], nan_policy="omit").correlation)

    de_top20 = float(np.nanmean(jcats)) if len(jcats) else np.nan
    de_rho   = float(np.nanmean(rhos))  if len(rhos)  else np.nan
    return de_top20, de_rho
